# Etapa 9 — Baselines Verdadeiramente Ingênuos

## Motivação

O *baseline autoregressivo* (`dumb_baseline.ipynb`) treina um XGBoost sobre 5 *features* de preço e atinge AUC = 0,658. Esse modelo, apesar de simples, **ainda aprende** — ele ajusta pesos, otimiza uma função de perda e explora relações não-lineares entre lags de preço e o target.

Para que a comparação metodológica deste capítulo seja completa, precisamos também de baselines que **não aprendem absolutamente nada**: modelos que ignoram todo e qualquer padrão nos dados de treino. Esses são os verdadeiros pisos teóricos de qualquer classificador.

Três baselines ingênuos são implementados:

| Baseline | Lógica | O que aprende? |
|---|---|---|
| **Classe majoritária** | sempre prevê P(sobe) = 1,0 | Nada — ignora todos os dados |
| **Coin flip ponderado** | prevê P(sobe) = prevalência no treino para toda amostra | Apenas a proporção de classes |
| **Persistência** | direção prevista = direção observada nos últimos HORIZON dias | Apenas a última observação |

Esses três baselines definem o **piso absoluto** de qualquer modelo de predição binária. Qualquer modelo que não os supere claramente não tem utilidade prática.

## Protocolo

- **Dataset**: `../2.stocks/dataset_full.csv` — o mesmo usado em todos os experimentos do Capítulo 5.
- **Target**: binário, `1` se `Close[t+21] > Close[t]`, senão `0` (HORIZON = 21 dias úteis).
- **Split**: walk-forward 70/15/15 idêntico ao dos experimentos anteriores.
- **Métricas**: ROC-AUC com **intervalo de confiança bootstrap 95%** (1.000 reamostragens), acurácia, F1 por classe.
- **Saída**: `results_naive_baselines.csv` com resumo de todas as métricas.

## 1. Imports e carregamento do dataset

In [1]:
import sys, os
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from eval_utils import (
    walk_forward_split, evaluate_model, make_binary_target,
    bootstrap_auc_ci, format_metric_with_ci,
)

DATASET_PATH = '../2.stocks/dataset_full.csv'
HORIZON = 21
SEED = 42
np.random.seed(SEED)

df = pd.read_csv(DATASET_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

# Construir target binário com horizonte de 21 dias
df['target'] = make_binary_target(df['Close'], horizon=HORIZON)
df = df.dropna(subset=['target']).reset_index(drop=True)

print(f'Dataset: {len(df)} dias | balance sobe={df["target"].mean():.3f} | desce={1 - df["target"].mean():.3f}')

Dataset: 1206 dias | balance sobe=0.590 | desce=0.410


## 2. Split walk-forward 70/15/15

Idêntico ao usado em todos os experimentos do Capítulo 5. O conjunto de treino é usado apenas para calcular a prevalência de classe (prior) — não há ajuste de parâmetros além disso.

In [2]:
train_df, val_df, test_df = walk_forward_split(df, train_frac=0.70, val_frac=0.15)
print(f'Train: {len(train_df)} ({train_df["Date"].min().date()} → {train_df["Date"].max().date()})')
print(f'Val:   {len(val_df)} ({val_df["Date"].min().date()} → {val_df["Date"].max().date()})')
print(f'Test:  {len(test_df)} ({test_df["Date"].min().date()} → {test_df["Date"].max().date()})')

y_train = train_df['target'].values.astype(int)
y_test  = test_df['target'].values.astype(int)

# Prior: prevalência da classe positiva (sobe) no treino
train_prior = float(y_train.mean())
print(f'\nPrior de treino (P(sobe)): {train_prior:.4f}')
print(f'Classe majoritária no treino: {"Sobe" if train_prior > 0.5 else "Desce"}')

Train: 844 (2021-04-28 → 2024-09-10)
Val:   180 (2024-09-11 → 2025-06-04)
Test:  182 (2025-06-05 → 2026-02-25)

Prior de treino (P(sobe)): 0.5699
Classe majoritária no treino: Sobe


## 3. Baseline A — Classe majoritária

**Lógica**: sempre prevê P(sobe) = 1,0, independentemente de qualquer *feature* ou padrão observado. É o classificador mais simples possível: emite sempre a classe dominante.

**AUC esperado**: como o modelo atribui a mesma pontuação a todas as amostras, a curva ROC é uma linha horizontal — **AUC teórico = 0,5**, independentemente do balanço de classes.

**Utilidade**: define o piso absoluto de AUC. Qualquer modelo com AUC < 0,5 é pior que não fazer nada.

In [3]:
# Sempre prevê P(sobe) = 1.0 para toda amostra do teste
y_score_majority = np.ones(len(y_test), dtype=float)

results_majority = evaluate_model(y_test, y_score_majority)
print('Classe majoritária — AUC:', results_majority['auc_str'])
print(f'Acurácia: {results_majority["accuracy"]:.3f}')
print(f'F1(sobe): {results_majority["f1_pos"]:.3f} | F1(desce): {results_majority["f1_neg"]:.3f}')
print(results_majority['report'])

Classe majoritária — AUC: 0.500 [0.500, 0.500]
Acurácia: 0.692
F1(sobe): 0.818 | F1(desce): 0.000
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        56
           1       0.69      1.00      0.82       126

    accuracy                           0.69       182
   macro avg       0.35      0.50      0.41       182
weighted avg       0.48      0.69      0.57       182



## 4. Baseline B — Coin flip ponderado

**Lógica**: prevê P(sobe) = prevalência da classe positiva no treino para **toda** amostra do teste. O modelo não discrimina entre amostras — ele emite uma probabilidade constante igual ao prior observado.

Diferentemente da classe majoritária (que prevê 1,0 para todas), este baseline emite uma probabilidade intermediária que reflete o desequilíbrio real da série.

**AUC esperado**: como todas as probabilidades previstas são idênticas, a curva ROC é também uma linha horizontal — **AUC teórico = 0,5**.

**Utilidade**: este baseline é importante para verificar se qualquer ganho de AUC observado em modelos reais é de fato discriminação, e não apenas calibração de probabilidade.

In [4]:
# Prevê P(sobe) = prior do treino para toda amostra
y_score_coin = np.full(len(y_test), fill_value=train_prior, dtype=float)

results_coin = evaluate_model(y_test, y_score_coin)
print(f'Coin flip ponderado (prior={train_prior:.4f}) — AUC:', results_coin['auc_str'])
print(f'Acurácia: {results_coin["accuracy"]:.3f}')
print(f'F1(sobe): {results_coin["f1_pos"]:.3f} | F1(desce): {results_coin["f1_neg"]:.3f}')
print(results_coin['report'])

Coin flip ponderado (prior=0.5699) — AUC: 0.500 [0.500, 0.500]
Acurácia: 0.692
F1(sobe): 0.818 | F1(desce): 0.000
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        56
           1       0.69      1.00      0.82       126

    accuracy                           0.69       182
   macro avg       0.35      0.50      0.41       182
weighted avg       0.48      0.69      0.57       182



## 5. Baseline C — Persistência

**Lógica**: "o futuro repete o passado imediato". Para cada dia `t` do conjunto de teste, a direção prevista é igual à direção observada na janela anterior de `HORIZON` dias: se `Close[t] > Close[t - HORIZON]`, prevê "Sobe" (P=1,0); caso contrário, prevê "Desce" (P=0,0).

Este baseline captura o raciocínio intuitivo de **momentum ingênuo**: a tendência de curto prazo continua. Em mercados com forte reversão à média, este baseline seria pior que o acaso; em mercados com momentum persistente, poderia ser competitivo.

**AUC esperado**: depende da autocorrelação da série — pode ser > 0,5 se houver momentum ou < 0,5 se houver reversão. Por isso é o mais informativo dos três baselines.

In [5]:
# Para calcular a persistência, precisamos do Close nos HORIZON dias anteriores ao início do teste
# O dataset completo está em df (já ordenado por data)

test_indices = test_df.index.tolist()  # índices no df original
close_series = df['Close'].values

y_score_persistence = np.empty(len(test_indices), dtype=float)

for i, idx in enumerate(test_indices):
    past_idx = idx - HORIZON
    if past_idx < 0:
        # Sem histórico suficiente: prevê classe majoritária
        y_score_persistence[i] = float(train_prior > 0.5)
    else:
        # Direção dos últimos HORIZON dias: 1 se subiu, 0 se caiu
        y_score_persistence[i] = 1.0 if close_series[idx] > close_series[past_idx] else 0.0

results_persistence = evaluate_model(y_test, y_score_persistence)
print('Persistência — AUC:', results_persistence['auc_str'])
print(f'Acurácia: {results_persistence["accuracy"]:.3f}')
print(f'F1(sobe): {results_persistence["f1_pos"]:.3f} | F1(desce): {results_persistence["f1_neg"]:.3f}')
print(results_persistence['report'])

Persistência — AUC: 0.474 [0.405, 0.543]
Acurácia: 0.560
F1(sobe): 0.688 | F1(desce): 0.259
              precision    recall  f1-score   support

           0       0.27      0.25      0.26        56
           1       0.68      0.70      0.69       126

    accuracy                           0.56       182
   macro avg       0.47      0.47      0.47       182
weighted avg       0.55      0.56      0.56       182



## 6. Tabela resumo e salvamento dos resultados

In [6]:
summary = pd.DataFrame([
    {
        'modelo': 'Classe majoritária',
        'descricao': 'Sempre prevê P(sobe)=1.0',
        'auc': results_majority['auc'],
        'auc_ci_lower': results_majority['auc_ci'][0],
        'auc_ci_upper': results_majority['auc_ci'][1],
        'auc_str': results_majority['auc_str'],
        'accuracy': results_majority['accuracy'],
        'f1_pos': results_majority['f1_pos'],
        'f1_neg': results_majority['f1_neg'],
        'n_test': results_majority['n_test'],
        'class_balance': results_majority['class_balance'],
    },
    {
        'modelo': 'Coin flip ponderado',
        'descricao': f'Prevê P(sobe)=prior_treino={train_prior:.4f} para toda amostra',
        'auc': results_coin['auc'],
        'auc_ci_lower': results_coin['auc_ci'][0],
        'auc_ci_upper': results_coin['auc_ci'][1],
        'auc_str': results_coin['auc_str'],
        'accuracy': results_coin['accuracy'],
        'f1_pos': results_coin['f1_pos'],
        'f1_neg': results_coin['f1_neg'],
        'n_test': results_coin['n_test'],
        'class_balance': results_coin['class_balance'],
    },
    {
        'modelo': 'Persistência',
        'descricao': f'Direção prevista = direção dos últimos {HORIZON} dias',
        'auc': results_persistence['auc'],
        'auc_ci_lower': results_persistence['auc_ci'][0],
        'auc_ci_upper': results_persistence['auc_ci'][1],
        'auc_str': results_persistence['auc_str'],
        'accuracy': results_persistence['accuracy'],
        'f1_pos': results_persistence['f1_pos'],
        'f1_neg': results_persistence['f1_neg'],
        'n_test': results_persistence['n_test'],
        'class_balance': results_persistence['class_balance'],
    },
])

summary.to_csv('results_naive_baselines.csv', index=False)
print('Resultados salvos em results_naive_baselines.csv')
print()
print(summary[['modelo', 'auc_str', 'accuracy', 'f1_pos', 'f1_neg']].to_string(index=False))

Resultados salvos em results_naive_baselines.csv

             modelo              auc_str  accuracy   f1_pos   f1_neg
 Classe majoritária 0.500 [0.500, 0.500]  0.692308 0.818182 0.000000
Coin flip ponderado 0.500 [0.500, 0.500]  0.692308 0.818182 0.000000
       Persistência 0.474 [0.405, 0.543]  0.560440 0.687500 0.259259


## 7. Interpretação e posicionamento na hierarquia de baselines

### O que estes números significam

Os três baselines ingênuos definem a **hierarquia de dificuldade** do problema de predição:

| Baseline | AUC esperado | Interpretação |
|---|---:|---|
| Classe majoritária | = 0,500 | Piso teórico absoluto — nenhum modelo pode ser pior em AUC |
| Coin flip ponderado | = 0,500 | Idêntico ao anterior em AUC; difere apenas na probabilidade emitida |
| Persistência | variável | Captura autocorrelação de curto prazo; pode ser > ou < 0,5 |

### Hierarquia completa de baselines (do mais ingênuo ao mais sofisticado)

```
AUC = 0,500  ← Classe majoritária / Coin flip (piso absoluto)
AUC ~ ???    ← Persistência (depende da série)
AUC = 0,658  ← Baseline autoregressivo XGBoost (5 features de preço) [Experimento 5.3]
AUC = 0,709  ← Transformer + FinBERT-PT-BR [resultado original, janela única]
```

### Interpretação do gap

- **Gap entre piso absoluto e baseline autoregressivo**: 0,658 − 0,500 = **0,158 pontos de AUC**. Este gap é o que o XGBoost extrai das relações não-lineares entre lags de preço.
- **Gap entre baseline autoregressivo e Transformer-FinBERT (janela única)**: 0,709 − 0,658 = **0,051 pontos de AUC**. Este é o "ganho" aparente do sentimento — pequeno e, como demonstrado nos experimentos 5.5 a 5.10, estatisticamente indistinguível de ruído amostral.
- **Gap entre persistência e baseline autoregressivo**: depende do resultado empírico acima; se persistência estiver próxima de 0,5, confirma que lags simples de preço *per se* não contêm sinal direcional trivial.

### Nota metodológica sobre AUC = 0,500 para baselines constantes

Os baselines de classe majoritária e coin flip emitirão AUC = 0,500 por construção (todos os scores são idênticos, logo a curva ROC é a diagonal). O `bootstrap_auc_ci` pode retornar CI degenerado neste caso (lower = upper = 0,5). Isso é matematicamente correto e esperado — **não é um erro**.